# Notebook 06 — Final Evaluation on Held-Out Test Set

**This notebook touches the test set for the first time.**

Until now every decision — hyperparameters, model choice, feature engineering — was made using only the train and val sets. The test set was locked away in Drive since Notebook 02. Evaluating on it now gives an honest, unbiased estimate of real-world performance.

**What this notebook produces:**
1. Final accuracy and macro F1 on the held-out test set
2. Per-class precision, recall, F1 — the full classification report
3. Normalised confusion matrix showing where errors concentrate
4. Error analysis — which classes are most confused and why
5. Confidence distribution — is the model well-calibrated?
6. Project summary table — ready for resume and slides

## Cell 1 — Mount Drive & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

PROJECT_DIR = '/content/drive/MyDrive/nlp_project'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
EMBED_DIR   = os.path.join(PROJECT_DIR, 'embeddings')
MODEL_DIR   = os.path.join(PROJECT_DIR, 'models')
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Ready.')

## Cell 2 — Load Model, Embeddings, Labels, and Mappings

In [ ]:
# Load the winning model saved in NB05
model = xgb.XGBClassifier()
model.load_model(os.path.join(MODEL_DIR, 'xgb_sbert_only.json'))
print('Model loaded: xgb_sbert_only.json')

# Load SBERT embeddings (encoded and cached in NB05)
emb_test = np.load(os.path.join(EMBED_DIR, 'sbert_test.npy'))
emb_val  = np.load(os.path.join(EMBED_DIR, 'sbert_val.npy'))   # for val vs test comparison
print(f'Test embeddings  : {emb_test.shape}')
print(f'Val  embeddings  : {emb_val.shape}')

# Load labels
df_test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
df_val  = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
y_test  = df_test['label'].values
y_val   = df_val['label'].values

# Load label mapping
with open(os.path.join(DATA_DIR, 'label_mapping.json')) as f:
    mapping  = json.load(f)
    id2label = {int(k): v for k, v in mapping['id2label'].items()}

CLASS_NAMES = [id2label[i] for i in sorted(id2label)]
N_CLASSES   = len(CLASS_NAMES)
print(f'Classes ({N_CLASSES}): {CLASS_NAMES}')

## Cell 3 — Run Predictions on Test Set

The SBERT model was trained on sparse `csr_matrix` embeddings.  
We convert the dense numpy array to the same format before predicting.

In [ ]:
# Convert to sparse — same format as training
X_test = sp.csr_matrix(emb_test.astype(np.float32))
X_val  = sp.csr_matrix(emb_val.astype(np.float32))

# Predictions and probabilities
y_pred_test  = model.predict(X_test)
y_proba_test = model.predict_proba(X_test)   # shape: (n_samples, n_classes)

y_pred_val   = model.predict(X_val)           # for val vs test comparison

print(f'Test predictions shape  : {y_pred_test.shape}')
print(f'Test probability shape  : {y_proba_test.shape}')
print(f'Unique predicted classes: {sorted(np.unique(y_pred_test))}')

## Cell 4 — Core Metrics

In [ ]:
acc_test      = accuracy_score(y_test, y_pred_test)
macro_f1_test = f1_score(y_test, y_pred_test, average='macro')
macro_p_test  = precision_score(y_test, y_pred_test, average='macro')
macro_r_test  = recall_score(y_test, y_pred_test, average='macro')

acc_val       = accuracy_score(y_val, y_pred_val)
macro_f1_val  = f1_score(y_val, y_pred_val, average='macro')

print('='*52)
print('  FINAL TEST SET RESULTS')
print('='*52)
print(f'  Accuracy         : {acc_test:.4f}  ({acc_test*100:.2f}%)')
print(f'  Macro F1         : {macro_f1_test:.4f}  ({macro_f1_test*100:.2f}%)')
print(f'  Macro Precision  : {macro_p_test:.4f}')
print(f'  Macro Recall     : {macro_r_test:.4f}')
print('='*52)
print(f'\nVal  Accuracy: {acc_val:.4f}  |  Val  Macro F1: {macro_f1_val:.4f}')
print(f'Test Accuracy: {acc_test:.4f}  |  Test Macro F1: {macro_f1_test:.4f}')
gap_acc = abs(acc_val - acc_test)
gap_f1  = abs(macro_f1_val - macro_f1_test)
print(f'\nVal→Test gap  Accuracy: {gap_acc:.4f}  |  Macro F1: {gap_f1:.4f}')
print('(Gap < 0.02 = no overfitting, gap > 0.05 = possible overfitting)')

## Cell 5 — Full Per-Class Report

In [ ]:
print('Per-class classification report (TEST SET):\n')
print(classification_report(y_test, y_pred_test, target_names=CLASS_NAMES, digits=3))

## Cell 6 — Normalised Confusion Matrix

In [ ]:
cm      = confusion_matrix(y_test, y_pred_test)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, vmin=0, vmax=1, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual',    fontsize=12)
ax.set_title('SBERT + XGBoost — Final Confusion Matrix (Test Set)', fontsize=13)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'final_confusion_matrix.png'), dpi=120)
plt.show()
print('Saved: final_confusion_matrix.png')

## Cell 7 — Error Analysis

For each class, we find the class it most often gets confused with.  
This tells us which product boundaries are semantically blurry.

In [ ]:
print('Error analysis — where does each class leak?\n')
print(f'{"True Class":<28} {"Correct":>8} {"Top mistake → predicted as":<35} {"Mistake %":>10}')
print('-' * 85)

for i, true_name in enumerate(CLASS_NAMES):
    row       = cm[i]
    correct   = cm[i, i]
    total     = row.sum()
    correct_r = correct / total

    # Top wrong prediction (exclude diagonal)
    wrong_counts = row.copy()
    wrong_counts[i] = 0
    top_wrong_idx   = wrong_counts.argmax()
    top_wrong_count = wrong_counts[top_wrong_idx]
    top_wrong_pct   = top_wrong_count / total

    if top_wrong_count > 0:
        print(f'{true_name:<28} {correct_r:>7.1%}   → {CLASS_NAMES[top_wrong_idx]:<35} {top_wrong_pct:>9.1%}')
    else:
        print(f'{true_name:<28} {correct_r:>7.1%}   → (no errors)')

## Cell 8 — Confidence Distribution

For each prediction, we look at the model's max predicted probability (its confidence).  
A well-calibrated model should be:
- High confidence on correct predictions
- Lower confidence on wrong predictions

In [ ]:
max_proba   = y_proba_test.max(axis=1)    # confidence per sample
is_correct  = (y_pred_test == y_test)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(max_proba[is_correct],  bins=40, alpha=0.7, color='steelblue', label=f'Correct  (n={is_correct.sum():,})')
ax.hist(max_proba[~is_correct], bins=40, alpha=0.7, color='coral',     label=f'Wrong    (n={(~is_correct).sum():,})')
ax.set_xlabel('Max predicted probability (confidence)')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution — Correct vs Wrong Predictions (Test Set)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confidence_distribution.png'), dpi=120)
plt.show()

print(f'Correct predictions  — mean confidence: {max_proba[is_correct].mean():.3f}')
print(f'Wrong predictions    — mean confidence: {max_proba[~is_correct].mean():.3f}')

## Cell 9 — Sample Predictions

A few real complaint examples with predicted vs actual label.  
Qualitative check: does the model make sensible errors?

In [ ]:
df_test['predicted'] = [id2label[p] for p in y_pred_test]
df_test['actual']    = [id2label[a] for a in y_test]
df_test['correct']   = df_test['predicted'] == df_test['actual']
df_test['confidence']= max_proba

# Show 2 correct + 2 wrong predictions
correct_samples = df_test[df_test['correct']].sample(2, random_state=1)
wrong_samples   = df_test[~df_test['correct']].sample(2, random_state=1)
samples = pd.concat([correct_samples, wrong_samples])

for _, row in samples.iterrows():
    status = '✓ CORRECT' if row['correct'] else '✗ WRONG'
    print(f'{status}  |  Actual: {row["actual"]}  |  Predicted: {row["predicted"]}  |  Confidence: {row["confidence"]:.3f}')
    print(f'  Text: {str(row["narrative_clean"])[:200]}...')
    print()

## Cell 10 — Project Summary Table

Final comparison across all models — this is your resume/slide table.

In [ ]:
# Per-class F1 on test
f1_per_test = f1_score(y_test, y_pred_test, average=None)

# Reference numbers from earlier notebooks (val set)
tfidf_f1_val  = [0.382, 0.457, 0.851, 0.150, 0.369, 0.345, 0.343]
tfidf_acc_val = 0.7118
tfidf_mf1_val = 0.4140

print('=' * 68)
print('  PROJECT RESULTS SUMMARY')
print('=' * 68)
print(f'{"":<30} {"TF-IDF (val)":>14} {"SBERT (test)":>14}')
print('-' * 68)
for i, name in enumerate(CLASS_NAMES):
    delta  = f1_per_test[i] - tfidf_f1_val[i]
    arrow  = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '→')
    print(f'  {name:<28} {tfidf_f1_val[i]:>14.3f} {f1_per_test[i]:>14.3f}  {arrow}{delta:+.3f}')
print('-' * 68)
print(f'  {"Macro F1":<28} {tfidf_mf1_val:>14.3f} {macro_f1_test:>14.3f}  ↑{macro_f1_test-tfidf_mf1_val:+.3f}')
print(f'  {"Accuracy":<28} {tfidf_acc_val:>14.3f} {acc_test:>14.3f}  ↑{acc_test-tfidf_acc_val:+.3f}')
print('=' * 68)
print(f'\n  Dataset  : CFPB Consumer Complaint Database')
print(f'  Classes  : {N_CLASSES} product categories')
print(f'  Train    : 139,771 complaints')
print(f'  Test     : {len(df_test):,} complaints (held-out, never seen during training)')

## Cell 11 — Final Summary (paste this output to Claude)

In [ ]:
print('='*60)
print('  NOTEBOOK 06 SUMMARY — paste this output to Claude')
print('='*60)
print(f'Final model          : SBERT (all-MiniLM-L6-v2) + XGBoost')
print(f'Evaluated on         : held-out test set ({len(df_test):,} samples)')
print()
print(f'Test Accuracy        : {acc_test:.4f}  ({acc_test*100:.2f}%)')
print(f'Test Macro F1        : {macro_f1_test:.4f}  ({macro_f1_test*100:.2f}%)')
print(f'Test Macro Precision : {macro_p_test:.4f}')
print(f'Test Macro Recall    : {macro_r_test:.4f}')
print()
print(f'Val  Macro F1        : {macro_f1_val:.4f}')
print(f'Val→Test gap         : {gap_f1:.4f}  (overfitting check)')
print()
print('Per-class F1 (test):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:<30} {f1_per_test[i]:.3f}')
print()
print(f'Correct predictions mean confidence : {max_proba[is_correct].mean():.3f}')
print(f'Wrong predictions mean confidence   : {max_proba[~is_correct].mean():.3f}')
print('='*60)